# reFuel.ch Data Ingestion

This notebook is the workflow-facing Step 1 ingestion pipeline for the MOTEL platform. The implementation lives in `ingestion_helper.py`; this notebook focuses on the process, the source data, and the resulting unmapped entities.

## Workflow

1. define run controls and repository paths
2. load the `unmapped_entity` schema and the reFuel.ch workbook
3. inspect the workbook sheets and mapping context
4. run the three sheet conversions
5. preview the outputs, export YAML files, and publish a combined staging file for harmonisation


## Run Controls

Use `TEST_SAMPLE_LIMIT` for quick testing and `PREVIEW_ROWS` to control how much of the converted output is previewed in the notebook.


In [1]:
TEST_SAMPLE_LIMIT = None
PREVIEW_ROWS = 3

In [2]:
import importlib

import scripts.ingestion_helper as ih
importlib.reload(ih)

PATHS = ih.get_refuel_paths()

print(f"Project root: {PATHS['project_root']}")
print(f"Workbook: {PATHS['workbook_path']}")
print(f"Schema: {PATHS['schema_path']}")
print(f"Combined staging output: {PATHS['staging_path']}")

Project root: E:\Barton\repositories\motel-platform
Workbook: E:\Barton\repositories\motel-platform\1_ingest\examples\refuel\input\reFuel_TechDatabase_Clean_2026-06-03.xlsx
Schema: E:\Barton\repositories\motel-platform\schema_human\unmapped_entity.yaml
Combined staging output: E:\Barton\repositories\motel-platform\motel-db\unmapped_entity\unmapped_entities_refuel.yaml


## Load Schema And Source Workbook

This stage makes the ingestion contract and the source workbook visible before any transformation is run.


In [3]:
schema_unmapped = ih.load_schema(PATHS["schema_path"])
refuel_data = ih.load_workbook(PATHS["workbook_path"])
context = ih.build_ingestion_context(refuel_data)

print("Top-level unmapped_entity sections:")
for key in schema_unmapped.get("properties", {}):
    print(f"- {key}")

print()
print("Workbook sheets:")
for sheet_name, df_sheet in refuel_data.items():
    print(f"- {sheet_name}: {len(df_sheet)} rows x {len(df_sheet.columns)} columns")

Top-level unmapped_entity sections:

Workbook sheets:
- Metadata: 62 rows x 10 columns
- Nomenclature: 124 rows x 6 columns
- Carrier: 69 rows x 7 columns
- ConvTech: 105 rows x 40 columns
- StorTech: 24 rows x 27 columns
- EmbeddedCarbon: 67 rows x 15 columns
- Reference: 44 rows x 6 columns


## Inspect Mapping Context

The conversion uses the `Reference` sheet for source provenance, the `Metadata` sheet for attribute-level notes and units, the `Nomenclature` sheet for controlled-vocabulary context, and the `Carrier` sheet for carrier notes.


In [4]:
df_ref = context["df_ref"]
df_attr = context["df_attr"]
df_nom = context["df_nom"]
df_carrier = context["df_carrier"]

print(f"Reference rows: {len(df_ref)}")
print(f"Mapped attributes: {len(df_attr)}")
print(f"Nomenclature rows: {len(df_nom)}")
print(f"Carrier rows: {len(df_carrier)}")
print()
print("Attributes used in this pipeline:")
for attr_name in df_attr.index.tolist():
    print(f"- {attr_name}")

df_attr.head()

Reference rows: 43
Mapped attributes: 27
Nomenclature rows: 124
Carrier rows: 69

Attributes used in this pipeline:
- reference_unit_size
- trl
- tech_maturity
- technical_efficiency
- theoretical_efficiency
- operating_temperature_c
- min_installation_size
- storage_carrier
- min_installation
- charging_capacity_factor
- discharging_capacity_factor
- charging_efficiency
- discharging_efficiency
- min_soc
- max_soc
- standby_loss_per_hour
- lifetime_yr
- capex_one_time
- capex_per_capacity
- capex_per_stor_capacity
- opex_one_time
- opex_fix_pct_of_capex
- opex_per_capacity_yr
- opex_per_stor_capacity_yr
- opex_per_energy
- discount_rate_pct
- uncertainty_rating


,Column Header,Unit / Format,Allowed Values,Description,Note
Variable Name,,,,,
reference_unit_size,Reference Unit Size,[MW] or [t/h],> 0,Nameplate capacity on which all specific cost ...,Capacity basis for scaling
trl,TRL,—,1–9,Technology Readiness Level on the standard 1–9...,See Nomenclature 4.2
tech_maturity,Technology Maturity,—,"{Prototype, Emerging, Mature}",Aggregated maturity classification derived fro...,See Nomenclature 4.2
technical_efficiency,Technical Efficiency,[0–1],0 < η ≤ 1,Overall energy conversion efficiency: main out...,"Based on kWh (electricity, heat) or LHV (chemi..."
theoretical_efficiency,Theoretical Efficiency,[0–1],0 < η ≤ 1,Ratio of the main output's energy content (LHV...,May be blank if not applicable


## Run The Sheet Conversions

`ConvTech` and `StorTech` share the same transformation logic. `EmbeddedCarbon` is handled separately because each row expands into multiple time-indexed `embedded_carbon` attributes.


In [5]:
sheet_entities = ih.run_refuel_pipeline(
    workbook=refuel_data,
    df_ref=df_ref,
    df_attr=df_attr,
    df_nom=df_nom,
    df_carrier=df_carrier,
    df_scope_meta=context["df_scope_meta"],
    sample_limit=TEST_SAMPLE_LIMIT,
)

for sheet_name, entities in sheet_entities.items():
    ih.preview_entities(sheet_name, entities, preview_rows=PREVIEW_ROWS)
    print()

ConvTech: using all 99 rows
Cannot find source_id "paper_biomass to syngas" in source dataset
Cannot find source_id "report_wood to hydrogen" in source dataset
Cannot find source_id "paper_biomass to syngas" in source dataset
Cannot find source_id "EP2050+Exkurs" in source dataset
StorTech: using all 22 rows
EmbeddedCarbon: using all 65 rows
ConvTech: 99 entities
- NH3_CCGT_El
- NH3_OCGT_El
- CH4_CCGT_El

StorTech: 22 entities
- ElBdg_Battery_ElBdg
- ElBdg_Battery_ElBdg
- ElBdg_Battery_ElBdg

EmbeddedCarbon: 65 entities
- Air_DAC_CO2
- BioWet_Digestion_CH4
- C12_Tank_C12



## Inspect Converted Records

Use this section to spot-check the unmapped entities before exporting them.


In [6]:
sheet_entities["ConvTech"][:PREVIEW_ROWS]

[{'technology_name': 'NH3_CCGT_El',
  'technology': {'technology_description': None,
   'technology_type': 'Conversion',
   'technology_category': 'Gas turbine',
   'technology_notes': 'technology_category = Technology class Gas turbine: Open or combined cycle gas turbine power generation',
   'process_name': 'CCGT',
   'process_type': None,
   'process_category': None,
   'process_notes': 'Input carriers: NH3 | Output carriers: El13'},
  'scope': {'geographic_scope': 'ECA',
   'geographic_scope_description': 'Europe and Central Asia; based on World Bank regional classification',
   'temporal_scope': '2050',
   'temporal_scope_description': None,
   'capacity_scope': None,
   'capacity_scope_description': None,
   'system_boundary': 'Plant ready to operate',
   'system_boundary_description': 'Default boundary. Equipment, installation, and essential auxiliary systems required for operation. Excludes external grid reinforcement and land acquisition.',
   'scope_notes': "geographic_scope 

## Export And Publish

The per-sheet YAML files are useful for inspection. The combined staging YAML in `motel-db/unmapped_entity/` is the handoff into `2_harmonise/2_data_harmonisation.ipynb`.


In [7]:
output_paths = {
    "ConvTech": PATHS["convtech_output"],
    "StorTech": PATHS["stortech_output"],
    "EmbeddedCarbon": PATHS["embeddedcarbon_output"],
}

ih.export_sheet_entities(sheet_entities, output_paths)
for sheet_name, output_path in output_paths.items():
    print(f"Exported {len(sheet_entities[sheet_name])} entities to {output_path.relative_to(PATHS['notebook_dir'])}")

published_entities = ih.publish_unmapped_entities(
    sheet_entities.values(),
    PATHS["staging_path"],
)

print()
print(f"Published {len(published_entities)} entities -> {PATHS['staging_path'].relative_to(PATHS['project_root'])}")
print("Mapping status for published records: to_be_mapped")

Exported 99 entities to output\unmapped_entities_refuel_convtech.yaml
Exported 22 entities to output\unmapped_entities_refuel_stortech.yaml
Exported 65 entities to output\unmapped_entities_refuel_embeddedcarbon.yaml

Published 186 entities -> motel-db\unmapped_entity\unmapped_entities_refuel.yaml
Mapping status for published records: to_be_mapped
